In [29]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_UNEMPLOYMENT_RATE")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_unemployment_links.parquet", index=False)

In [5]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 106


In [6]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\(\s*(\[\[.*?\]\])\s*\);'

sleep_duration = 5

In [7]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [8]:
# DataFrames de salida
dfs_quarters = []
dfs_months = []

start_time = time.time()
for url in country_links:
    country = urlparse(url).path.strip("/").split("/")[0]

    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            # --- separar según formato de Period ---
            if df["Period"].str.contains(r"Q\d").any():
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_quarters.append(df_pivot)
                print(f"✅ Datos trimestrales extraídos para {country}")
            else:
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_months.append(df_pivot)
                print(f"✅ Datos mensuales extraídos para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

# unir cada grupo por separado
df_quarters = pd.concat(dfs_quarters, axis=1) if dfs_quarters else pd.DataFrame()
df_months = pd.concat(dfs_months, axis=1) if dfs_months else pd.DataFrame()

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Andorra no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
✅ Datos mensuales extraídos para Australia
✅ Datos mensuales extraídos para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos mensuales extraídos para Belgium
⏭️ País Bolivia no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
✅ Datos mensuales extraídos para Brazil
✅ Datos mensuales extraídos para Bulgaria
✅ Datos mensuales extraídos para Canada
✅ Datos mensuales extraídos para Chile
✅ Datos trimestrales extraídos para China
✅ Datos mensuales extraídos para Colombia
⏭️ País Costa-Rica no está en la lista, se omite.
✅ Datos mensuales extraídos para Croatia
✅ 

In [9]:
# resultados finales
print("\n📊 DataFrame trimestral")
df_quarters.head()


📊 DataFrame trimestral


Country,China,Macedonia,Morocco,Philippines,Serbia,Singapore,South-Africa,Thailand
Period,,,,,,,,
Q1 2015,4.10,27.40,9.9,6.40,19.0,1.8,26.4,0.94
Q1 2016,4.04,24.50,10.0,5.60,19.0,1.9,26.7,0.97
Q1 2017,3.97,22.95,10.7,5.70,14.6,2.2,27.7,1.21
Q1 2018,3.89,21.61,10.5,5.50,14.8,2.0,26.7,1.24
Q1 2019,3.67,17.83,9.1,5.13,12.1,2.2,27.6,0.92


In [10]:
print("\n📊 DataFrame mensual")
df_months.head()


📊 DataFrame mensual


Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,Croatia,Cyprus,...,Russia,Slovakia,Slovenia,South-Korea,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1978,6.25,NaN,NaN,NaN,NaN,9.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.1,5.6
Apr 1979,6.46,NaN,NaN,NaN,NaN,8.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8,5.3
Apr 1980,6.15,NaN,NaN,NaN,NaN,8.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.9,6.1
Apr 1981,5.58,NaN,NaN,NaN,NaN,7.7,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.2,9.4
Apr 1982,6.40,NaN,NaN,NaN,NaN,10.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.3,10.5


In [11]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [12]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [13]:
df_paraguay.tail()

Country,Paraguay
Period,
Q4 2020,7.17
Q4 2021,6.81
Q4 2022,5.80
Q4 2023,5.23
Q4 2024,4.55


In [14]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
new_quarter_df = pd.merge(df_quarters, df_paraguay, how="outer", left_index=True, right_index=True)
new_quarter_df.head()

Country,China,Macedonia,Morocco,Philippines,Serbia,Singapore,South-Africa,Thailand,Paraguay
Period,,,,,,,,,
Q1 1992,NaN,NaN,NaN,NaN,NaN,1.8,NaN,NaN,NaN
Q1 1993,NaN,NaN,NaN,NaN,NaN,1.5,NaN,NaN,NaN
Q1 1994,NaN,NaN,NaN,NaN,NaN,1.7,NaN,NaN,NaN
Q1 1995,NaN,NaN,NaN,NaN,NaN,1.7,NaN,NaN,NaN
Q1 1996,NaN,NaN,NaN,NaN,NaN,1.8,NaN,NaN,NaN


In [15]:
def split_period_index(df):
    """
    Recibe un DataFrame con índice 'Period' (ej: 'Apr 1960' o 'Q1 1990')
    Devuelve el mismo DataFrame con columnas 'Year' y 'Month' o 'Quarter'
    """
    df = df.copy().reset_index()  # sacar Period del índice
    df["Year"] = None
    df["Month"] = None
    df["Quarter"] = None

    for i, period in enumerate(df["Period"]):
        parts = period.split()

        if len(parts) == 2:
            first, year = parts
            df.at[i, "Year"] = int(year)

            if "Q" in first:  # Formato trimestral
                df.at[i, "Quarter"] = first
            else:  # Formato mensual
                df.at[i, "Month"] = first

    return df.set_index("Period")  # opcional: devolver con Period otra vez de índice

In [16]:
new_quarter_df = split_period_index(new_quarter_df)
new_quarter_df.head()

Country,China,Macedonia,Morocco,Philippines,Serbia,Singapore,South-Africa,Thailand,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,
Q1 1992,NaN,NaN,NaN,NaN,NaN,1.8,NaN,NaN,NaN,1992,None,Q1
Q1 1993,NaN,NaN,NaN,NaN,NaN,1.5,NaN,NaN,NaN,1993,None,Q1
Q1 1994,NaN,NaN,NaN,NaN,NaN,1.7,NaN,NaN,NaN,1994,None,Q1
Q1 1995,NaN,NaN,NaN,NaN,NaN,1.7,NaN,NaN,NaN,1995,None,Q1
Q1 1996,NaN,NaN,NaN,NaN,NaN,1.8,NaN,NaN,NaN,1996,None,Q1


In [18]:
new_month_df = split_period_index(df_months)
new_month_df.head()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,Croatia,Cyprus,...,South-Korea,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1978,6.25,NaN,NaN,NaN,NaN,9.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.1,5.6,1978,Apr,None
Apr 1979,6.46,NaN,NaN,NaN,NaN,8.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5.8,5.3,1979,Apr,None
Apr 1980,6.15,NaN,NaN,NaN,NaN,8.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.9,6.1,1980,Apr,None
Apr 1981,5.58,NaN,NaN,NaN,NaN,7.7,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.2,9.4,1981,Apr,None
Apr 1982,6.40,NaN,NaN,NaN,NaN,10.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,9.3,10.5,1982,Apr,None


In [19]:
def month_to_quarter(df):
    """
    Takes a DataFrame with 'Month' and 'Quarter' columns.
    If 'Month' exists, it assigns the corresponding quarter to 'Quarter'.
    """
    df = df.copy()

    # Dictionary for mapping months → quarters
    month_to_q = {
        "Jan": 1, "Feb": 1, "Mar": 1,
        "Apr": 2, "May": 2, "June": 2,
        "Jul": 3, "Aug": 3, "Sep": 3,
        "Oct": 4, "Nov": 4, "Dec": 4,
    }

    # Update quarter only where Month exists
    df["Quarter"] = df.apply(
        lambda row: month_to_q.get(row["Month"], row["Quarter"]),
        axis=1
    )

    return df

In [20]:
def quarter_to_int(df):
    """
    Convierte los valores de la columna 'Quarter' a enteros (Q1 → 1, Q2 → 2, etc.)
    """
    df = df.copy()

    q_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}

    df["Quarter"] = df["Quarter"].map(q_map).fillna(df["Quarter"])

    return df

In [21]:
def year_to_int(df):
    """
    Convierte la columna 'Year' a enteros.
    Si hay valores no numéricos, los convierte en NaN.
    """
    df = df.copy()
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

In [23]:
new_month_df.head()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,Croatia,Cyprus,...,South-Korea,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1978,6.25,NaN,NaN,NaN,NaN,9.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.1,5.6,1978,Apr,2
Apr 1979,6.46,NaN,NaN,NaN,NaN,8.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5.8,5.3,1979,Apr,2
Apr 1980,6.15,NaN,NaN,NaN,NaN,8.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.9,6.1,1980,Apr,2
Apr 1981,5.58,NaN,NaN,NaN,NaN,7.7,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.2,9.4,1981,Apr,2
Apr 1982,6.40,NaN,NaN,NaN,NaN,10.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,9.3,10.5,1982,Apr,2


In [26]:
new_month_df = month_to_quarter(new_month_df)
new_month_df = year_to_int(new_month_df)
new_month_df.dtypes

Country
Australia         float64
Austria           float64
Belgium           float64
Brazil            float64
Bulgaria          float64
Canada            float64
Chile             float64
Colombia          float64
Croatia           float64
Cyprus            float64
Czech-Republic    float64
Denmark           float64
Estonia           float64
Finland           float64
France            float64
Germany           float64
Greece            float64
Hong-Kong         float64
Hungary           float64
Iceland           float64
India             float64
Ireland           float64
Israel            float64
Italy             float64
Japan             float64
Latvia            float64
Lithuania         float64
Luxembourg        float64
Malaysia          float64
Malta             float64
Mexico            float64
Netherlands       float64
New-Zealand       float64
Norway            float64
Poland            float64
Portugal          float64
Romania           float64
Russia            float64
Slov

In [27]:
new_quarter_df = quarter_to_int(new_quarter_df)
new_quarter_df = year_to_int(new_quarter_df)
new_quarter_df.dtypes

Country
China           float64
Macedonia       float64
Morocco         float64
Philippines     float64
Serbia          float64
Singapore       float64
South-Africa    float64
Thailand        float64
Paraguay        float64
Year              Int64
Month            object
Quarter           int64
dtype: object

In [28]:
# Guardar como archivo Parquet
new_month_df.to_parquet("../../data/month_unemployment_values.parquet", index=False)
new_quarter_df.to_parquet("../../data/quarter_unemployment_values.parquet", index=False)